# Ground truth and LLM-as-judge scoring (v2: hybrid judge)

**Part A** builds the cleaned ground truth (no GPU server needed).
**Part B** scores one predictions file with **one judge**. Run it once per judge: start that judge's vLLM server, change `JUDGE_MODEL` in one cell, run Part B.
**Part C** combines the three judges, reports their agreement and compares systems (needs the verdict files of Part B, no server).

**Privacy:** the saved outputs of this notebook can contain patient-derived text. Clear all outputs before sharing.

# Part A. Ground truth

## A1. Configuration

In [ ]:
import json, re, collections
from pathlib import Path
import pandas as pd

PATIENT_DIR = Path("patient_contexts")
TARGET_DIR  = Path("patient_targets")
DEV_CASE    = "patient_00000000_00000000.txt"   # tuning case, excluded from the test set
GT_FILE     = Path("ground_truth_dx.json")
GT_REPORT   = Path("ground_truth_report.json")
FLAGS_FILE  = Path("expire_flags.csv")

## A2. Parse the Discharge Diagnosis list and apply the cleaning rules

Rules, applied to every case and every model in the same way (write these in the methods):
1. Items that only state the patient's death (e.g. "Expired", "Deceased", "pt expired") or a code-status / care-limitation (CMO, comfort measures, DNR, DNI) are removed.
2. Items without any letters, and bare headings ending in a colon (e.g. "PMH:"), are removed.
3. Placeholders such as "N/A" are removed.
4. A case with no diagnosis section, or no items left, is not scorable and is excluded from scoring (for every model).

In [ ]:
HEAD_END = r"(?:Discharge Condition|Discharge Instructions|Condition at Discharge|Mental Status|Level of Consciousness|Activity Status|Follow-?up(?: Instructions)?)"

def parse_discharge_dx(text):
    m = re.search(r"Discharge Diagnos[ie]s\s*:\s*\n?(.*?)(?=\n\s*" + HEAD_END + r"\s*:|\Z)", text, re.I | re.S)
    if not m:
        return None
    out = {"primary": [], "secondary": [], "other": []}
    cur = "other"
    for line in m.group(1).splitlines():
        s = line.strip().strip("-*\u2022").strip()
        s = re.sub(r"^\d+[.)]\s*", "", s)
        if not s:
            continue
        low = s.lower().rstrip(":").strip()
        if re.match(r"^primary( diagnos[ie]s)?$", low):
            cur = "primary"; continue
        if re.match(r"^secondary( diagnos[ie]s)?$", low):
            cur = "secondary"; continue
        out[cur].append(s)
    return out

DROP = re.compile(r"^[^A-Za-z]*(?:the\s+)?(?:(?:patient|pt)\.?\s+)?(?:is\s+|was\s+|has\s+)?(?:expired|deceased|dead|died|death)\b(?!.*\b(?:of|from|due|secondary|after|following|with|d/t)\b).*$", re.I)
CARE = re.compile(r"\b(CMO|comfort measures|comfort care|DNR|DNI)\b", re.I)
JUNK_WORDS = {"n/a", "na", "none", "nil", "unknown"}

def is_junk(x):
    return (not re.search(r"[A-Za-z]", x)) or bool(re.match(r"^\s*[A-Za-z /]{1,20}:\s*$", x))

def clean_items(items):
    kept, removed = [], collections.Counter()
    for x in items:
        if DROP.match(x) or CARE.search(x):
            removed["death_or_care_limitation"] += 1
        elif is_junk(x):
            removed["no_letters_or_heading"] += 1
        elif x.strip().lower().strip(".") in JUNK_WORDS:
            removed["placeholder_na"] += 1
        else:
            kept.append(x)
    return kept, removed

In [ ]:
cases = [f for f in sorted(p.name for p in PATIENT_DIR.glob("patient_*.txt")) if f != DEV_CASE]

gt, no_section, emptied, removed_total = {}, [], [], collections.Counter()
for fname in cases:
    path = TARGET_DIR / fname.replace(".txt", "_target.txt")
    dx = parse_discharge_dx(path.read_text(encoding="utf-8"))
    items = (dx["primary"] + dx["secondary"] + dx["other"]) if dx else []
    if not items:
        no_section.append(fname); continue
    kept, removed = clean_items(items)
    removed_total.update(removed)
    if kept:
        gt[fname] = kept
    else:
        emptied.append(fname)

if GT_FILE.exists():
    old = json.load(open(GT_FILE, encoding="utf-8"))
    if old != gt:
        Path("ground_truth_dx_prev.json").write_text(json.dumps(old, indent=2), encoding="utf-8")
        print("Existing ground truth differed - previous version saved as ground_truth_dx_prev.json")
json.dump(gt, open(GT_FILE, "w", encoding="utf-8"), indent=2)

sizes = sorted(len(v) for v in gt.values())
report = {"cases_considered": len(cases), "no_diagnosis_section": len(no_section),
          "nothing_left_after_cleaning": len(emptied), "scorable_cases": len(gt),
          "items_removed": dict(removed_total),
          "items_per_case_min_median_max": [sizes[0], sizes[len(sizes)//2], sizes[-1]]}
json.dump(report, open(GT_REPORT, "w"), indent=2)
print(json.dumps(report, indent=2))
print("Earlier manual run for comparison: 99 considered, 3 no section, 11 emptied, 85 scorable;",
      "removed 11 death/care + 22 junk + 2 N/A.")

## A3. Death flags (analysis only, never shown to any model)

In [ ]:
if FLAGS_FILE.exists():
    df_flags = pd.read_csv(FLAGS_FILE)
    print("Loaded", FLAGS_FILE)
else:
    from google.cloud import bigquery
    bq = bigquery.Client(project="my-dissertation-project-502620")
    hadm = {f: int(re.match(r"patient_\d+_(\d+)\.txt$", f).group(1))
            for f in sorted(p.name for p in PATIENT_DIR.glob("patient_*.txt"))}
    q = """
    SELECT hadm_id, hospital_expire_flag
    FROM `physionet-data.mimiciv_3_1_hosp.admissions`
    WHERE hadm_id IN UNNEST(@ids)
    """
    cfg = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("ids", "INT64", list(hadm.values()))])
    flags = bq.query(q, job_config=cfg).to_dataframe()
    by_hadm = dict(zip(flags["hadm_id"], flags["hospital_expire_flag"]))
    df_flags = pd.DataFrame({"file_name": list(hadm.keys())})
    df_flags["died_in_hospital"] = [int(by_hadm.get(h, -1)) for h in hadm.values()]
    df_flags.to_csv(FLAGS_FILE, index=False)
    print("Queried BigQuery and saved", FLAGS_FILE)

died = dict(zip(df_flags["file_name"], df_flags["died_in_hospital"]))
print("all cases      - died:", int((df_flags["died_in_hospital"] == 1).sum()), "| survived:", int((df_flags["died_in_hospital"] == 0).sum()))
print("scorable cases - died:", sum(int(died[f]) for f in gt), "| survived:", sum(1 - int(died[f]) for f in gt))
print("excluded cases - died:", sum(int(died[f]) for f in no_section + emptied), "of", len(no_section + emptied))

# Part B. Score one predictions file with one judge

## B1. Configuration

**To switch judge:** stop the current vLLM server, start the next judge's server, change **only `JUDGE_MODEL`**, then run Part B again.
The notebook checks that the server really is serving `JUDGE_MODEL`, so verdicts can never be saved under the wrong name.

In [ ]:
import time, datetime
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

JUDGE_MODEL = "allenai/Olmo-3-7B-Instruct"  #"ibm-granite/granite-4.1-8b"  #"meta-llama/Llama-3.1-8B-Instruct"   # <-- the ONLY line to change when switching judge
PRED_FILE   = Path("predictions_model1_27b.json")      # predictions to score
PRED_TAG    = "model1_27b"                             # used in the output file name
OVERWRITE   = False
MAX_PRED    = 15                                   # only the first MAX_PRED ranked predictions of each case are scored (same rule for every model)

SERVER_URL  = "http://localhost:8000/v1"
JUDGE_TAG   = JUDGE_MODEL.split("/")[-1].lower()
OUT_FILE    = Path(f"judge_v2_{PRED_TAG}_{JUDGE_TAG}.json")

client = OpenAI(base_url=SERVER_URL, api_key="x")
served = [m.id for m in client.models.list().data]
assert JUDGE_MODEL in served, f"Server is serving {served}, not {JUDGE_MODEL}. Start that judge's vLLM server first."
print("Judge server OK:", served, "| output file:", OUT_FILE)

## B2. Judge design (v2, hybrid)

The matching rules agreed with the domain expert are split into two parts so that an 8B model is only asked one simple question:

1. **Acute / chronic check (code, no LLM).** Each term is tagged ACUTE, CHRONIC, ACUTE_ON_CHRONIC or NONE (from the words "acute"/"chronic" and a few abbreviations such as AKI, CKD, COPD). The expert's table is applied exactly: a chronic reference rejects acute and acute-on-chronic predictions; an acute-on-chronic reference accepts an acute-only prediction and rejects a chronic-only one; a term with no acuity stated is not blocked.
2. **Same-disease question (LLM).** Ignoring acute/chronic wording: accept the same disease, synonyms, abbreviations, less specific forms that still name the disease (site, side, organism, cause or severity dropped) and more specific forms; reject broad categories ("infection", "cancer") and merely related conditions.

A pair is a match only if both parts agree. The examples inside the prompt are deliberately different from the pairs in the known-answer panel below. **Frozen: the wording was approved by the supervisor before the real scoring runs. Do not edit it.**

*Confirmed:* an "acute" reference with an "acute on chronic" prediction is accepted (symmetric with the expert's rule).

In [ ]:
import re

# ---------- Part 1: acute / chronic check (code) ----------
ACUTE_ABBR   = {"aki", "arf", "ards", "atn"}              # acute kidney injury, acute renal failure, ARDS, acute tubular necrosis
CHRONIC_ABBR = {"ckd", "esrd", "eskd", "crf", "copd"}     # chronic kidney disease, end-stage renal/kidney disease, chronic renal failure, COPD

def acuity_tag(term):
    words = set(re.findall(r"[a-z]+", term.lower()))
    has_acute   = "acute" in words or bool(words & ACUTE_ABBR)
    has_chronic = "chronic" in words or bool(words & CHRONIC_ABBR)
    if has_acute and has_chronic:
        return "ACUTE_ON_CHRONIC"
    if has_acute:
        return "ACUTE"
    if has_chronic:
        return "CHRONIC"
    return "NONE"

def acuity_ok(ref_tag, pred_tag):
    """Expert's rules. NONE on either side is never blocked."""
    if ref_tag == "NONE" or pred_tag == "NONE" or ref_tag == pred_tag:
        return True
    allowed = {
        ("ACUTE_ON_CHRONIC", "ACUTE"),   # expert: acute-only prediction is accepted
        ("ACUTE", "ACUTE_ON_CHRONIC"),   # confirmed: treated symmetrically with the expert's rule
    }
    return (ref_tag, pred_tag) in allowed    # every other mix of acute/chronic is a NO

# ---------- Part 2: same-disease question (LLM) ----------
JUDGE_RULE = """Do these two medical terms name the same disease or condition?
Term A is the reference diagnosis. Term B is a predicted diagnosis.
Ignore the words "acute", "chronic" and "acute on chronic": they are checked separately.

Answer YES if Term B names the same specific disease as Term A:
- the same wording, a synonym, or a standard medical abbreviation (for example MI = myocardial infarction);
- a less specific form that drops a qualifier such as severity, type, site, side, organism or cause
  but still names the disease (for example "pneumonia" for "right upper lobe pneumonia",
  "sepsis" for "E. coli sepsis", "bloodstream infection" for "MRSA bloodstream infection");
- a more specific form (for example "type 1 diabetes mellitus" for "diabetes").

Answer NO if:
- Term B is only a broad category covering many different diseases and does not name the specific
  disease (for example "cancer" for "pancreatic adenocarcinoma", "heart disease" for "atrial fibrillation");
  a term that still names the organ or site is enough (for example "bloodstream infection");
- Term B is only related to Term A: a cause, a complication, a different organ system,
  or a different condition.

Reply with exactly one word: YES or NO."""

def llm_same_disease(a, b):
    r = client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0, max_tokens=3,
        messages=[{"role": "user", "content": f"{JUDGE_RULE}\n\nTerm A: {a}\nTerm B: {b}"}],
    )
    return r.choices[0].message.content.strip().upper().startswith("YES")

def pair_verdict(a, b):
    if not acuity_ok(acuity_tag(a), acuity_tag(b)):
        return False                     # blocked by the acute/chronic rule, no LLM call needed
    return llm_same_disease(a, b)

def judge_case(gt_items, pred_items, workers=16):
    """Matrix M[i][j]: does prediction j (rank j+1) match ground-truth item i?"""
    pairs = [(i, j) for i in range(len(gt_items)) for j in range(len(pred_items))]
    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(lambda ij: pair_verdict(gt_items[ij[0]], pred_items[ij[1]]), pairs))
    M = [[False] * len(pred_items) for _ in gt_items]
    for (i, j), v in zip(pairs, res):
        M[i][j] = v
    return M

def recall_at_k(M, k):
    return sum(any(row[:k]) for row in M) / len(M) if M else None

KS = (1, 3, 5, 10, 15)

def summarise(verdicts):
    for k in KS:
        vals = [recall_at_k(M, k) for M in verdicts.values()]
        print(f"mean recall@{k}: {sum(vals)/len(vals):.3f}")

### B2b. Unit tests for the code part (no LLM, no patient data). Expect all passed.

In [ ]:
TAG_TESTS = [("Acute kidney injury", "ACUTE"), ("AKI", "ACUTE"), ("Chronic kidney disease", "CHRONIC"),
             ("CKD", "CHRONIC"), ("COPD", "CHRONIC"), ("Acute on chronic systolic heart failure", "ACUTE_ON_CHRONIC"),
             ("Acute exacerbation of COPD", "ACUTE_ON_CHRONIC"), ("AKI on CKD", "ACUTE_ON_CHRONIC"),
             ("Heart failure", "NONE"), ("Pneumonia", "NONE")]
RULE_TESTS = [("CHRONIC", "ACUTE_ON_CHRONIC", False), ("CHRONIC", "ACUTE", False), ("ACUTE", "CHRONIC", False),
              ("ACUTE_ON_CHRONIC", "CHRONIC", False), ("ACUTE_ON_CHRONIC", "ACUTE", True),
              ("ACUTE_ON_CHRONIC", "NONE", True), ("NONE", "ACUTE", True), ("CHRONIC", "CHRONIC", True)]
t_ok = sum(acuity_tag(t) == want for t, want in TAG_TESTS)
r_ok = sum(acuity_ok(a, b) == want for a, b, want in RULE_TESTS)
print("acuity tag tests passed:", t_ok, "of", len(TAG_TESTS), "| rule-table tests passed:", r_ok, "of", len(RULE_TESTS))
for t, want in TAG_TESTS:
    if acuity_tag(t) != want: print("  tag mismatch:", t, "->", acuity_tag(t), "expected", want)

## B3. Known-answer panel (sanity check of the judge)

Made-up pairs (no patient data) with known answers, covering the matching rules agreed with the domain expert. Expect 18 of 18; below 17 means this judge needs a closer look before it is used. The acute/chronic pairs are now decided by code, so they should always pass.

In [ ]:
PANEL = [
    # same or equivalent -> YES
    ("Acute kidney injury", "AKI", True),
    ("Heart failure", "Acute systolic heart failure", True),
    ("Diabetes", "Type 2 diabetes mellitus", True),
    # less specific (qualifier dropped, same disease) -> YES
    ("Left lower lobe pneumonia", "Pneumonia", True),
    ("Acute on chronic systolic heart failure", "Heart failure", True),
    ("Serratia CNS infection", "CNS infection", True),
    # abbreviations -> YES
    ("Pneumonia", "PNA", True),
    ("Hyperlipidemia", "HLD", True),
    # acute on chronic reference
    ("Acute on chronic systolic heart failure", "Acute systolic heart failure", True),
    ("Acute on chronic systolic heart failure", "Chronic systolic heart failure", False),
    ("Chronic systolic heart failure", "Acute on chronic systolic heart failure", False),
    # acute versus chronic -> NO
    ("Acute kidney injury", "Chronic kidney disease", False),
    ("Chronic kidney disease", "Acute kidney injury", False),
    # too vague / broad category -> NO
    ("Serratia CNS infection", "Infection", False),
    ("Acute kidney injury", "Kidney disease", False),
    # related but different -> NO
    ("Pneumonia", "Urinary tract infection", False),
    ("Hypertension", "Hypotension", False),
    ("Anemia", "Thrombocytopenia", False),
]
ok = 0
for a, b, want in PANEL:
    got = pair_verdict(a, b)
    ok += (got == want)
    print(f"{a} | {b} -> {'YES' if got else 'NO'} (expected {'YES' if want else 'NO'})")
print("known-answer panel correct:", ok, "of", len(PANEL))

## B4. Run the judge on the scorable cases and save the verdicts

In [ ]:
gt    = json.load(open(GT_FILE, encoding="utf-8"))
preds = json.load(open(PRED_FILE, encoding="utf-8"))
cases = [f for f in preds if f in gt]
n_cut = sum(1 for f in cases if len(preds[f]["diagnoses"]) > MAX_PRED)
print("predicted:", len(preds), "| scorable (in ground truth):", len(cases), f"| lists longer than {MAX_PRED} (cut): {n_cut}")

if OUT_FILE.exists() and not OVERWRITE:
    print(OUT_FILE, "already exists - set OVERWRITE = True to redo it.")
    verdicts = json.load(open(OUT_FILE, encoding="utf-8"))["verdicts"]
else:
    t0, verdicts = time.time(), {}
    for fname in cases:
        verdicts[fname] = judge_case(gt[fname], preds[fname]["diagnoses"][:MAX_PRED])
    meta = {"judge_model": JUDGE_MODEL, "predictions_file": str(PRED_FILE), "ground_truth_file": str(GT_FILE),
            "n_cases": len(verdicts), "max_pred": MAX_PRED, "lists_cut": n_cut, "date": datetime.datetime.now().isoformat(timespec="seconds"),
            "judge_rule": JUDGE_RULE, "design": "v2 hybrid: rule-based acute/chronic check in code + LLM same-disease question"}
    json.dump({"meta": meta, "verdicts": verdicts}, open(OUT_FILE, "w", encoding="utf-8"))
    print("cases judged:", len(verdicts), "| seconds:", round(time.time() - t0), "| saved:", OUT_FILE)

summarise(verdicts)

# Part C. Combine the three judges and compare systems

Run Part B once per judge and per predictions file first (six runs for two systems). Then run Part C, which needs no server.

Following the supervisor's guidance: for each ground-truth item and each k, the three judges' verdicts ("was this item found within the first k predictions?") are **averaged at item level first**, then averaged within each case, then across cases. A majority-vote version is reported as a sensitivity check, together with the agreement between judges (Krippendorff's alpha, nominal).

## C1. Configuration and loading

In [ ]:
import json, random, itertools
from pathlib import Path
from collections import defaultdict

JUDGE_TAGS = ["llama-3.1-8b-instruct", "granite-4.1-8b", "olmo-3-7b-instruct"]   # JUDGE_TAG values used in Part B
SYSTEMS    = {"MedGemma 4B": "model1", "MedGemma 27B": "model1_27b"}              # name -> PRED_TAG used in Part B
KS = (1, 3, 5, 10, 15)

def found_at(M, k):
    return [any(row[:k]) for row in M]          # one boolean per ground-truth item

verdicts = {}                                    # verdicts[system][judge][case] = matrix
for sysname, ptag in SYSTEMS.items():
    verdicts[sysname] = {}
    for jt in JUDGE_TAGS:
        f = Path(f"judge_v2_{ptag}_{jt}.json")
        if not f.exists():
            print("MISSING:", f); continue
        verdicts[sysname][jt] = json.load(open(f, encoding="utf-8"))["verdicts"]

for sysname, d in verdicts.items():
    sets = [set(v) for v in d.values()]
    print(sysname, "| judges loaded:", len(d), "| cases per judge:", [len(x) for x in sets],
          "| identical case sets:", all(x == sets[0] for x in sets) if sets else None)

## C2. Item-level averaging across the three judges, then recall per case

In [ ]:
def per_case_recall(sysname, k, mode="mean"):
    """mode='mean': average the judges' verdicts per item (supervisor's guidance); mode='majority': majority vote."""
    d = verdicts[sysname]
    judges = list(d)
    cases = sorted(set.intersection(*[set(d[j]) for j in judges]))
    out = {}
    for c in cases:
        per_judge = [found_at(d[j][c], k) for j in judges]            # judges x items
        items = list(zip(*per_judge))                                  # items x judges
        if mode == "mean":
            scores = [sum(x) / len(x) for x in items]
        else:
            scores = [1.0 if sum(x) > len(x) / 2 else 0.0 for x in items]
        out[c] = sum(scores) / len(scores)
    return out

summary = {}
for sysname in verdicts:
    if len(verdicts[sysname]) < 2:
        print("skipping", sysname, "(fewer than 2 judges)"); continue
    summary[sysname] = {k: per_case_recall(sysname, k) for k in KS}
    row = "  ".join(f"@{k}: {sum(v.values())/len(v):.3f}" for k, v in summary[sysname].items())
    n = len(summary[sysname][15])
    print(f"{sysname:14s} n={n}  mean recall (judges averaged)  {row}")
for sysname in summary:
    row = "  ".join(f"@{k}: {sum(per_case_recall(sysname, k, 'majority').values())/len(summary[sysname][k]):.3f}" for k in KS)
    print(f"{sysname:14s} (majority-vote version)             {row}")

## C3. Agreement between the judges (Krippendorff's alpha, nominal) and per-judge recall

In [ ]:
def krippendorff_alpha_nominal(units):
    """units: list of lists of 0/1 ratings (one list per item, one rating per judge, no missing values)."""
    o = {(0, 0): 0.0, (0, 1): 0.0, (1, 0): 0.0, (1, 1): 0.0}
    for r in units:
        m = len(r)
        if m < 2:
            continue
        for a in range(m):
            for b in range(m):
                if a != b:
                    o[(r[a], r[b])] += 1.0 / (m - 1)
    n0, n1 = o[(0, 0)] + o[(0, 1)], o[(1, 0)] + o[(1, 1)]
    n = n0 + n1
    if n < 2 or n0 == 0 or n1 == 0:
        return float("nan")
    d_obs = (o[(0, 1)] + o[(1, 0)]) / n
    d_exp = 2 * n0 * n1 / (n * (n - 1))
    return 1 - d_obs / d_exp

for sysname in summary:
    d = verdicts[sysname]; judges = list(d)
    cases = sorted(set.intersection(*[set(d[j]) for j in judges]))
    for k in (5, 15):
        units = []
        for c in cases:
            units += [list(map(int, x)) for x in zip(*[found_at(d[j][c], k) for j in judges])]
        raw = sum(len(set(u)) == 1 for u in units) / len(units)
        print(f"{sysname:14s} found@{k:<2d}: items={len(units)}  raw agreement={raw:.3f}  alpha={krippendorff_alpha_nominal(units):.3f}")
    for j in judges:
        rec = sum(sum(found_at(d[j][c], 15)) / len(d[j][c]) for c in cases) / len(cases)
        print(f"   {j:24s} recall@15 on its own: {rec:.3f}")

## C4. Paired comparison of two systems on the same cases

Per-case recall difference (A minus B), a paired sign-flip permutation test on the mean difference, and a bootstrap 95% confidence interval. Both runs use the same seeds. Report the number of cases, the mean difference with its interval, and the p-value.

In [ ]:
def paired_compare(a_name, b_name, k=15, n_boot=10000, seed=1234):
    A, B = summary[a_name][k], summary[b_name][k]
    cases = sorted(set(A) & set(B))
    diffs = [A[c] - B[c] for c in cases]
    n = len(diffs); mean_d = sum(diffs) / n
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(diffs) for _ in range(n)) / n for _ in range(n_boot))
    lo, hi = boots[int(0.025 * n_boot)], boots[int(0.975 * n_boot) - 1]
    obs, extreme = abs(mean_d), 0
    for _ in range(n_boot):
        if abs(sum(d if rng.random() < 0.5 else -d for d in diffs) / n) >= obs - 1e-12:
            extreme += 1
    p_perm = (extreme + 1) / (n_boot + 1)
    try:
        from scipy.stats import wilcoxon
        p_w = wilcoxon(diffs).pvalue if any(d != 0 for d in diffs) else float("nan")
    except Exception:
        p_w = float("nan")
    print(f"recall@{k}: {a_name} {sum(A[c] for c in cases)/n:.3f} vs {b_name} {sum(B[c] for c in cases)/n:.3f} "
          f"| n={n} | mean diff {mean_d:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}) | permutation p={p_perm:.4f} | Wilcoxon p={p_w:.4f}")
    return mean_d, p_perm

names = list(summary)
if len(names) >= 2:
    res = {k: paired_compare(names[1], names[0], k=k) for k in (15, 5)}
    # Backbone selection rule (as written in the methods): higher mean recall@15 on the same cases wins,
    # unless the paired test is not significant, in which case the smaller model is kept.
    d15, p15 = res[15]
    print("Backbone rule applied: ", "larger model" if (d15 > 0 and p15 < 0.05) else "smaller model",
          "(larger model needs a positive difference with p < 0.05)")

In [ ]:
import pandas as pd
rows = []
for s in summary:
    for k in KS:
        mean_v = summary[s][k]
        maj_v  = per_case_recall(s, k, "majority")
        rows.append({"system": s, "k": k, "n_cases": len(mean_v),
                     "recall_judges_averaged": round(sum(mean_v.values()) / len(mean_v), 4),
                     "recall_majority_vote": round(sum(maj_v.values()) / len(maj_v), 4)})
pd.DataFrame(rows).to_csv("results_model1_recall.csv", index=False)
print("Saved results_model1_recall.csv |", len(rows), "rows")